# 03 - TRAINING MODELS

Mục tiêu:
- Huấn luyện nhiều mô hình hồi quy
- Sử dụng cùng một preprocessing pipeline
- So sánh các mô hình
- Thử nghiệm tham số
- Chuẩn bị mô hình tốt nhất cho bước đánh giá

In [1]:
import pandas as pd
import numpy as np
import joblib
import time

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.dummy import DummyRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Import thư viện thành công!")

Import thư viện thành công!


In [4]:
df = pd.read_csv("/content/winequality-red.csv")

df = df.drop_duplicates().reset_index(drop=True)

X = df.drop(columns=["quality"])
y = df["quality"]

print("Dataset:", df.shape)
print("X:", X.shape)
print("y:", y.shape)

Dataset: (1359, 12)
X: (1359, 11)
y: (1359,)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (1087, 11)
X_test : (272, 11)
y_train: (1087,)
y_test : (272,)


In [12]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

preprocessing_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

print("Preprocessing pipeline đã sẵn sàng!")
print(preprocessing_pipeline)

Preprocessing pipeline đã sẵn sàng!
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


In [13]:
preprocessing_pipeline.fit(X_train)

print("Đã fit pipeline!")

Đã fit pipeline!


In [16]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "Linear Regression": LinearRegression(),

    "Ridge Regression": Ridge(),

    "Random Forest": RandomForestRegressor(
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}

print("Đã tạo", len(models), "mô hình.")

for name in models:
    print("-", name)

Đã tạo 4 mô hình.
- Linear Regression
- Ridge Regression
- Random Forest
- Gradient Boosting


In [17]:
from sklearn.pipeline import Pipeline

model_pipelines = {}

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocess", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])),
        ("model", model)
    ])

    model_pipelines[name] = pipeline

print("Đã tạo pipeline cho tất cả mô hình.")

Đã tạo pipeline cho tất cả mô hình.


In [18]:
for name, pipeline in model_pipelines.items():
    print("\n", name)
    print(pipeline)


 Linear Regression
Pipeline(steps=[('preprocess',
                 Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                 ('scaler', StandardScaler())])),
                ('model', LinearRegression())])

 Ridge Regression
Pipeline(steps=[('preprocess',
                 Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                 ('scaler', StandardScaler())])),
                ('model', Ridge())])

 Random Forest
Pipeline(steps=[('preprocess',
                 Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                 ('scaler', StandardScaler())])),
                ('model', RandomForestRegressor(random_state=42))])

 Gradient Boosting
Pipeline(steps=[('preprocess',
                 Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                 ('scaler', StandardScaler())])),
                ('model', GradientBoostingRegressor(random_state

In [19]:
import time
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = []
trained_models = {}

for name, pipeline in model_pipelines.items():

    print(f"\n========== {name} ==========")

    # Đo thời gian train
    start_time = time.time()

    pipeline.fit(X_train, y_train)

    train_time = time.time() - start_time

    # Dự đoán
    start_time = time.time()

    y_pred = pipeline.predict(X_test)

    predict_time = time.time() - start_time

    # Tính metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Lưu kết quả
    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "Train Time (s)": train_time,
        "Predict Time (s)": predict_time
    })

    trained_models[name] = pipeline

    print(f"MAE : {mae:.4f}")
    print(f"MSE : {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²  : {r2:.4f}")


========== Linear Regression ==========
MAE : 0.5041
MSE : 0.4310
RMSE: 0.6565
R²  : 0.3915

========== Ridge Regression ==========
MAE : 0.5041
MSE : 0.4308
RMSE: 0.6564
R²  : 0.3918

========== Random Forest ==========
MAE : 0.4682
MSE : 0.3839
RMSE: 0.6196
R²  : 0.4580

========== Gradient Boosting ==========
MAE : 0.4791
MSE : 0.3835
RMSE: 0.6193
R²  : 0.4586


In [20]:
results_df = pd.DataFrame(results)

results_df

,Model,MAE,MSE,RMSE,R2,Train Time (s),Predict Time (s)
0,Linear Regression,0.504141,0.431009,0.656513,0.391536,0.068221,0.016586
1,Ridge Regression,0.504091,0.430831,0.656377,0.391788,0.062045,0.015213
2,Random Forest,0.468199,0.383912,0.619606,0.458024,1.704455,0.015243
3,Gradient Boosting,0.479145,0.383501,0.619275,0.458603,0.348975,0.003714


In [21]:
results_df_sorted = results_df.sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)

results_df_sorted

,Model,MAE,MSE,RMSE,R2,Train Time (s),Predict Time (s)
0,Gradient Boosting,0.479145,0.383501,0.619275,0.458603,0.348975,0.003714
1,Random Forest,0.468199,0.383912,0.619606,0.458024,1.704455,0.015243
2,Ridge Regression,0.504091,0.430831,0.656377,0.391788,0.062045,0.015213
3,Linear Regression,0.504141,0.431009,0.656513,0.391536,0.068221,0.016586


In [22]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean")

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_mse = mean_squared_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(baseline_mse)
baseline_r2 = r2_score(y_test, baseline_pred)

print("========== BASELINE ==========")
print("Dummy Regressor")
print(f"MAE : {baseline_mae:.4f}")
print(f"MSE : {baseline_mse:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"R²  : {baseline_r2:.4f}")

========== BASELINE ==========
Dummy Regressor
MAE : 0.7049
MSE : 0.7128
RMSE: 0.8443
R²  : -0.0063


In [23]:
from sklearn.model_selection import GridSearchCV

ridge_pipeline = Pipeline([
    ("preprocess", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])),
    ("model", Ridge())
])

ridge_params = {
    "model__alpha": [0.01, 0.1, 1, 10, 100]
}

ridge_grid = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_params,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

ridge_grid.fit(X_train, y_train)

print("Best parameters:")
print(ridge_grid.best_params_)

print("\nBest CV RMSE:")
print(-ridge_grid.best_score_)

Best parameters:
{'model__alpha': 10}

Best CV RMSE:
0.6648366020296004


In [24]:
rf_pipeline = Pipeline([
    ("preprocess", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])),
    ("model", RandomForestRegressor(
        random_state=42
    ))
])

rf_params = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_params,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("Best parameters:")
print(rf_grid.best_params_)

print("\nBest CV RMSE:")
print(-rf_grid.best_score_)

Best parameters:
{'model__max_depth': 10, 'model__min_samples_split': 2, 'model__n_estimators': 200}

Best CV RMSE:
0.6583386155755425


In [25]:
ridge_trials = pd.DataFrame(ridge_grid.cv_results_)

ridge_trials = ridge_trials[
    ["param_model__alpha", "mean_test_score", "std_test_score", "rank_test_score"]
].copy()

ridge_trials["Model"] = "Ridge Regression"
ridge_trials["CV_RMSE"] = -ridge_trials["mean_test_score"]

ridge_trials = ridge_trials.drop(
    columns=["mean_test_score"]
)

ridge_trials

,param_model__alpha,std_test_score,rank_test_score,Model,CV_RMSE
0,0.01,0.029983,4,Ridge Regression,0.665062
1,0.10,0.029979,3,Ridge Regression,0.665059
2,1.00,0.029934,2,Ridge Regression,0.665030
3,10.00,0.029571,1,Ridge Regression,0.664837
4,100.00,0.028554,5,Ridge Regression,0.665767


In [26]:
rf_trials = pd.DataFrame(rf_grid.cv_results_)

rf_trials = rf_trials[
    [
        "param_model__n_estimators",
        "param_model__max_depth",
        "param_model__min_samples_split",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].copy()

rf_trials["Model"] = "Random Forest"
rf_trials["CV_RMSE"] = -rf_trials["mean_test_score"]

rf_trials = rf_trials.drop(
    columns=["mean_test_score"]
)

rf_trials

,param_model__n_estimators,param_model__max_depth,param_model__min_samples_split,std_test_score,rank_test_score,Model,CV_RMSE
0,100,None,2,0.026541,5,Random Forest,0.659217
1,200,None,2,0.027073,6,Random Forest,0.659429
2,100,None,5,0.025804,8,Random Forest,0.660604
3,200,None,5,0.027339,7,Random Forest,0.660154
4,100,5,2,0.027292,9,Random Forest,0.661203
5,200,5,2,0.028388,10,Random Forest,0.661981
6,100,5,5,0.026969,12,Random Forest,0.662063
7,200,5,5,0.027894,11,Random Forest,0.662020
8,100,10,2,0.026741,3,Random Forest,0.658845
9,200,10,2,0.028021,1,Random Forest,0.658339


In [30]:
gb_pipeline = Pipeline([
    ("preprocess", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])),
    ("model", GradientBoostingRegressor(
        random_state=42
    ))
])

gb_params = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.05, 0.1],
    "model__max_depth": [2, 3]
}

gb_grid = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_params,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

gb_grid.fit(X_train, y_train)

print("Best parameters:")
print(gb_grid.best_params_)

print("\nBest CV RMSE:")
print(-gb_grid.best_score_)

Best parameters:
{'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100}

Best CV RMSE:
0.6570530973032764


In [31]:
gb_trials = pd.DataFrame(gb_grid.cv_results_)

In [32]:
gb_trials = pd.DataFrame(gb_grid.cv_results_)

gb_trials = gb_trials[
    [
        "param_model__n_estimators",
        "param_model__learning_rate",
        "param_model__max_depth",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].copy()

gb_trials["Model"] = "Gradient Boosting"
gb_trials["CV_RMSE"] = -gb_trials["mean_test_score"]

gb_trials = gb_trials.drop(
    columns=["mean_test_score"]
)

gb_trials

,param_model__n_estimators,param_model__learning_rate,param_model__max_depth,std_test_score,rank_test_score,Model,CV_RMSE
0,100,0.05,2,0.027050,2,Gradient Boosting,0.658562
1,200,0.05,2,0.028312,3,Gradient Boosting,0.666937
2,100,0.05,3,0.030177,1,Gradient Boosting,0.657053
3,200,0.05,3,0.035435,4,Gradient Boosting,0.667406
4,100,0.10,2,0.030560,5,Gradient Boosting,0.668528
5,200,0.10,2,0.036374,7,Gradient Boosting,0.679378
6,100,0.10,3,0.035808,6,Gradient Boosting,0.669800
7,200,0.10,3,0.040900,8,Gradient Boosting,0.679581


In [33]:
tuned_models = {
    "Ridge Tuned": ridge_grid.best_estimator_,
    "Random Forest Tuned": rf_grid.best_estimator_,
    "Gradient Boosting Tuned": gb_grid.best_estimator_
}

tuned_results = []

for name, model in tuned_models.items():

    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    start = time.time()
    y_pred = model.predict(X_test)
    predict_time = time.time() - start

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    tuned_results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "Train Time (s)": train_time,
        "Predict Time (s)": predict_time
    })

tuned_results_df = pd.DataFrame(tuned_results)

tuned_results_df = tuned_results_df.sort_values(
    by="RMSE"
)

tuned_results_df

,Model,MAE,MSE,RMSE,R2,Train Time (s),Predict Time (s)
1,Random Forest Tuned,0.468880,0.379509,0.616043,0.464240,1.727693,0.032850
2,Gradient Boosting Tuned,0.480592,0.385366,0.620779,0.455971,0.395465,0.005441
0,Ridge Tuned,0.503789,0.429545,0.655396,0.393603,0.013233,0.003848


In [34]:
# Kết hợp kết quả mô hình ban đầu và mô hình tuned

all_results = pd.concat(
    [
        results_df,
        tuned_results_df
    ],
    ignore_index=True
)

all_results = all_results.sort_values(
    by="RMSE"
).reset_index(drop=True)

all_results

,Model,MAE,MSE,RMSE,R2,Train Time (s),Predict Time (s)
0,Random Forest Tuned,0.468880,0.379509,0.616043,0.464240,1.727693,0.032850
1,Gradient Boosting,0.479145,0.383501,0.619275,0.458603,0.348975,0.003714
2,Random Forest,0.468199,0.383912,0.619606,0.458024,1.704455,0.015243
3,Gradient Boosting Tuned,0.480592,0.385366,0.620779,0.455971,0.395465,0.005441
4,Ridge Tuned,0.503789,0.429545,0.655396,0.393603,0.013233,0.003848
5,Ridge Regression,0.504091,0.430831,0.656377,0.391788,0.062045,0.015213
6,Linear Regression,0.504141,0.431009,0.656513,0.391536,0.068221,0.016586
